In [21]:
from enum import Enum
import torch
import timeit
import statistics
from cs336_basics.model import BasicsTransformerLM
from cs336_basics.optimizer import AdamW
from cs336_basics.data import get_batch
from cs336_basics.nn_utils import cross_entropy, clip_gradient


In [2]:
print(torch.cuda.get_device_name(0))

free, total = torch.cuda.mem_get_info()

print(f"Free:  {free / 1024**3:.2f} GB")
print(f"Total: {total / 1024**3:.2f} GB")
print(f"Used:  {(total - free) / 1024**3:.2f} GB")    


NVIDIA GeForce RTX 3070
Free:  7.38 GB
Total: 7.65 GB
Used:  0.27 GB


In [3]:
vocab_size = 10000
context_length = 512

FORWARD = 'forward'
BACKWARD = 'backward'
FULL = 'full'

In [4]:
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
import numpy as np

dataset = np.random.randint(
    low=0,
    high=vocab_size,
    size=(20000,),
    dtype=np.int64,
)

In [7]:
print(dataset)

[1981 4167 6608 ... 1255 5289 7975]


In [8]:

from dataclasses import dataclass

@dataclass
class BenchmarkConfig:
    d_model: int
    d_ff: int
    num_layers: int
    num_heads: int
    batch_size: int
    context_length: int
    warmup_iterations: int
    iterations: int
    vocab_size: int
    max_norm: float

In [9]:
def run_step(lm, optimizer, train_data, labels, mode):
    if mode != FORWARD:
        optimizer.zero_grad()

    logits = lm(train_data)
    loss = cross_entropy(logits, labels)

    if mode != FORWARD:
        loss.backward()

    if mode == FULL:
        optimizer.step()

In [ ]:
def benchmark_model(dataset, config: BenchmarkConfig,  mode, device):
    lm = BasicsTransformerLM(config.vocab_size, config.context_length, config.d_model, config.num_layers, config.num_heads, config.d_ff, rope_theta=None)
    param_bytes = sum(
        p.numel() * p.element_size()
        for p in lm.parameters()
    )

    print(f"Parameters: {param_bytes / 1024**3:.2f} GiB")

    lm = lm.to(device)
    optimizer = AdamW(lm.parameters())

    train_data, labels = get_batch(dataset, config.batch_size, context_length, device)
        # warmup
    for _ in range(config.warmup_iterations):
        run_step(lm, optimizer, train_data, labels, mode)

    torch.cuda.synchronize()

    # measurement
    times = []

    for _ in range(config.iterations):
        start = timeit.default_timer()

        run_step(lm, optimizer, train_data, labels, mode)

        torch.cuda.synchronize()
        times.append(timeit.default_timer() - start)
    return times

In [15]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [26]:
model_configs = {
    'small': BenchmarkConfig(d_model=768, d_ff=3072, num_layers=12, num_heads=12, batch_size=4, context_length=512, warmup_iterations=5, iterations=10, vocab_size=10000, max_norm=1.0),
    'medium': BenchmarkConfig(d_model=1024, d_ff=4096, num_layers=24, num_heads=16, batch_size=4, context_length=512, warmup_iterations=5, iterations=10, vocab_size=10000, max_norm=1.0),
    'large': BenchmarkConfig(d_model=1280, d_ff=5120, num_layers=36, num_heads=20, batch_size=4, context_length=512, warmup_iterations=5, iterations=10, vocab_size=10000, max_norm=1.0),
    'xl': BenchmarkConfig(d_model=2560, d_ff=10240, num_layers=32, num_heads=32, batch_size=4, context_length=512, warmup_iterations=5, iterations=10, vocab_size=10000, max_norm=1.0),
    '10B': BenchmarkConfig(d_model=4608, d_ff=12288, num_layers=50, num_heads=36, batch_size=4, context_length=512, warmup_iterations=5, iterations=10, vocab_size=10000, max_norm=1.0),
            
}

In [27]:
for name, model_config in model_configs.items():
    for mode in [FORWARD, BACKWARD, FULL]:
        small_times = benchmark_model(dataset, model_config, mode, device=device)
        print(name, mode)
        mean = statistics.mean(small_times)
        std = statistics.stdev(small_times)
        print(mean, std)

/home/abel/projects/assignment2-systems/cs336-basics/cs336_basics/model.py:361: UserWarning: No positional encoder provided
  self.attn = CausalMultiHeadSelfAttention(


small forward
0.0754362723999293 0.00018734082259515348
small backward
0.2411957888998586 0.0005367919115495145
small full
0.2866020313998888 0.0007279086519691463


OutOfMemoryError: CUDA out of memory. Tried to allocate 64.00 MiB. GPU 0 has a total capacity of 7.65 GiB of which 18.00 MiB is free. Including non-PyTorch memory, this process has 7.51 GiB memory in use. Of the allocated memory 7.17 GiB is allocated by PyTorch, and 146.54 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

10